# Introducción a números complejos I

Este notebook desarrolla de forma autoexplicativa los fundamentos de números complejos necesarios para álgebra lineal compleja y computación cuántica elemental. El énfasis está en la interpretación geométrica, el cálculo reproducible y la conexión entre amplitudes complejas y probabilidades.

La notación de Dirac se escribe de forma compatible con JupyterLab, Anaconda y Google Colab. Se usan expresiones LaTeX completas como

$$
\left|0\right\rangle,\qquad \left\langle \psi \right|,\qquad \left\langle \phi\middle|\psi\right\rangle.
$$

No se requieren macros personalizadas para que las fórmulas se rendericen correctamente.

## 1. Objetivos de aprendizaje

Al terminar este notebook podrás representar números complejos en forma rectangular, polar y exponencial; interpretar conjugado, módulo y argumento en el plano complejo; normalizar vectores complejos; calcular productos internos; aplicar la regla de Born; verificar matrices unitarias; y usar Python/NumPy/Matplotlib para visualizar y comprobar los resultados.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cmath
from math import sqrt, pi, atan2

plt.rcParams["figure.figsize"] = (6.5, 4.8)
plt.rcParams["axes.grid"] = True


## 2. Números complejos y plano complejo

Un número complejo tiene la forma

$$
z=a+b\,i,
$$

donde $a$ es la parte real, $b$ la parte imaginaria e $i^2=-1$. Geométricamente, $z$ se representa como el punto $(a,b)$ del plano. Esta representación permite interpretar el módulo como distancia al origen y el argumento como ángulo respecto al eje real positivo.

In [ ]:
def plot_complex_points(points, labels=None, title="Plano complejo"):
    """Dibuja números complejos como vectores desde el origen."""
    labels = labels or [str(z) for z in points]
    fig, ax = plt.subplots()
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    for z, label in zip(points, labels):
        ax.arrow(0, 0, z.real, z.imag, length_includes_head=True,
                 head_width=0.08, head_length=0.12)
        ax.scatter([z.real], [z.imag])
        ax.text(z.real + 0.08, z.imag + 0.08, label)
    max_val = max(1.0, max(abs(z.real) + 0.5 for z in points), max(abs(z.imag) + 0.5 for z in points))
    ax.set_xlim(-max_val, max_val)
    ax.set_ylim(-max_val, max_val)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Parte real")
    ax.set_ylabel("Parte imaginaria")
    ax.set_title(title)
    plt.show()

z = 2 + 3j
plot_complex_points([z], ["z = 2 + 3i"], "Representación de un complejo")

## 3. Conjugación compleja

El conjugado de $z=a+bi$ es

$$
\overline{z}=a-bi.
$$

Geométricamente, la conjugación refleja el punto respecto al eje real. Algebraicamente, permite formar la cantidad real no negativa $z\overline z=|z|^2$.

In [ ]:
z = 2 + 3j
zc = z.conjugate()
print("z =", z)
print("conjugado =", zc)
print("z * conjugado =", z * zc)
plot_complex_points([z, zc], ["z", "conjugado"], "Conjugación como reflexión")

## 4. Módulo y argumento

El módulo de $z=a+bi$ es

$$
|z|=\sqrt{a^2+b^2}.
$$

El argumento es un ángulo $\theta$ que localiza la dirección del vector. Para evitar errores de cuadrante se usa `atan2(b,a)` o `cmath.phase(z)`.

In [ ]:
z = 2 - 3j
r = abs(z)
theta = cmath.phase(z)
print("módulo:", r)
print("argumento principal:", theta)
plot_complex_points([z], ["2 - 3i"], "Número en el cuarto cuadrante")

## 5. Forma polar y fórmula de Euler

La forma polar de un complejo no nulo es

$$
z=r(\cos\theta+i\sin\theta).
$$

La fórmula de Euler permite escribirla de manera compacta:

$$
e^{i\theta}=\cos\theta+i\sin\theta,\qquad z=re^{i\theta}.
$$

In [ ]:
theta_values = np.linspace(0, 2*np.pi, 200)
circle = np.exp(1j * theta_values)
fig, ax = plt.subplots()
ax.plot(circle.real, circle.imag)
for th in [0, np.pi/3, np.pi/2, np.pi, 3*np.pi/2]:
    point = np.exp(1j * th)
    ax.scatter(point.real, point.imag)
    ax.text(point.real + 0.05, point.imag + 0.05, f"{th/np.pi:.2g}π")
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Círculo unidad y exponencial compleja")
ax.set_xlabel("Parte real")
ax.set_ylabel("Parte imaginaria")
plt.show()

## 6. Ejemplo: convertir $2e^{i\pi/3}$ a forma rectangular

Usando Euler:

$$
2e^{i\pi/3}=2\left(\cos\frac{\pi}{3}+i\sin\frac{\pi}{3}\right)=1+\sqrt{3}\,i.
$$

El resultado indica que el vector tiene módulo $2$, ángulo $60^\circ$, coordenada real $1$ y coordenada imaginaria $\sqrt3$.

In [ ]:
z = 2 * np.exp(1j * np.pi/3)
print(z)
print("parte real:", np.real(z))
print("parte imaginaria:", np.imag(z))
plot_complex_points([z], [r"$2e^{i\pi/3}$"], "Forma exponencial a rectangular")

## 7. Suma y producto como geometría

La suma de complejos desplaza puntos como vectores. El producto, en cambio, combina escala y orientación. Si

$$
z_1=r_1e^{i\theta_1},\qquad z_2=r_2e^{i\theta_2},
$$

entonces

$$
z_1z_2=r_1r_2e^{i(\theta_1+\theta_2)}.
$$

In [ ]:
z1 = 1 + 1j
z2 = 1j
product = z1 * z2
plot_complex_points([z1, z2, product], ["1+i", "i", "(1+i)i"], "Multiplicación por i como giro de 90 grados")

## 8. Vectores complejos y norma

Un vector complejo en $\mathbb C^n$ es una lista ordenada de complejos. Su norma euclidiana se define como

$$
\|v\|=\sqrt{\sum_i |v_i|^2}.
$$

Esta definición es la que permite hablar de estados cuánticos normalizados.

In [ ]:
v = np.array([1 + 1j, 2], dtype=complex)
norm_v = np.sqrt(np.vdot(v, v).real)
print("v =", v)
print("norma =", norm_v)
print("vector normalizado =", v / norm_v)
print("norma del normalizado =", np.vdot(v/norm_v, v/norm_v))

## 9. Producto interno complejo

Para $u,v\in\mathbb C^n$, usamos

$$
\langle u,v\rangle=\sum_i \overline{u_i}v_i.
$$

La conjugación del primer argumento garantiza que $\langle v,v\rangle=\sum_i |v_i|^2$ sea real y no negativa.

In [ ]:
u = np.array([1, 1j], dtype=complex)
v = np.array([1j, 1], dtype=complex)
print("np.vdot(u, v) =", np.vdot(u, v))
print("np.vdot(v, v) =", np.vdot(v, v))

## 10. Kets, bras y conjugado transpuesto

En notación de Dirac, un ket es un vector columna. Si

$$
\left|\psi\right\rangle=\begin{pmatrix}\alpha\\\beta\end{pmatrix},
$$

entonces su bra asociado es

$$
\left\langle\psi\right|=\begin{pmatrix}\overline{\alpha}&\overline{\beta}\end{pmatrix}.
$$

In [ ]:
psi = np.array([1 + 2j, 3 - 1j], dtype=complex)
bra = psi.conj().T
print("ket psi =", psi)
print("bra psi =", bra)
print("<psi|psi> =", bra @ psi)

## 11. Qubit y amplitudes complejas

Un qubit puro se representa por un vector normalizado en $\mathbb C^2$:

$$
\left|\psi\right\rangle=\alpha\left|0\right\rangle+\beta\left|1\right\rangle,
\qquad |\alpha|^2+|\beta|^2=1.
$$

Las amplitudes $\alpha$ y $\beta$ son complejas; las probabilidades se obtienen con módulos cuadrados.

In [ ]:
psi = np.array([(2-1j)/3, (np.sqrt(3)+1j)/3], dtype=complex)
probs = np.abs(psi)**2
print("amplitudes:", psi)
print("probabilidades:", probs)
print("suma:", probs.sum())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].axhline(0, linewidth=1); axes[0].axvline(0, linewidth=1)
labels = [r"$\alpha$ para $\left|0\right\rangle$", r"$\beta$ para $\left|1\right\rangle$"]
for amp, label in zip(psi, labels):
    axes[0].arrow(0, 0, amp.real, amp.imag, length_includes_head=True, head_width=0.03)
    axes[0].scatter(amp.real, amp.imag)
    axes[0].text(amp.real + 0.03, amp.imag + 0.03, label)
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_title("Amplitudes en el plano complejo")
axes[0].set_xlabel("Parte real"); axes[0].set_ylabel("Parte imaginaria")
axes[1].bar(["0", "1"], probs)
axes[1].set_ylim(0, 1)
axes[1].set_title("Probabilidades de medición")
axes[1].set_ylabel("Probabilidad")
plt.tight_layout()
plt.show()

## 12. Regla de Born

Si

$$
\left|\psi\right\rangle=\alpha\left|0\right\rangle+\beta\left|1\right\rangle,
$$

entonces

$$
\Pr(0)=|\alpha|^2,\qquad \Pr(1)=|\beta|^2.
$$

La probabilidad de observar $\left|0\right\rangle$ en el estado anterior es $|(2-i)/3|^2=5/9$.

In [ ]:
p0 = abs(psi[0])**2
print("P(0) =", p0)
print("fracción esperada: 5/9 =", 5/9)

## 13. Probabilidad de un estado base en dos qubits

Considera

$$
\left|\psi\right\rangle=-\frac{2i}{\sqrt6}\left|01\right\rangle+
\frac{1-i}{\sqrt6}\left|11\right\rangle.
$$

La probabilidad de observar $\left|11\right\rangle$ es el módulo cuadrado de su coeficiente:

$$
\left|\frac{1-i}{\sqrt6}\right|^2=\frac{2}{6}=\frac13.
$$

In [ ]:
amp_11 = (1 - 1j) / np.sqrt(6)
print(abs(amp_11)**2)

## 14. Matrices unitarias

Una matriz $U$ es unitaria si

$$
U^\dagger U=I.
$$

Esta condición garantiza que la norma de cualquier vector se conserve, por lo que las probabilidades totales siguen sumando uno.

In [ ]:
H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
for name, U in [("H", H), ("Y", Y), ("Z", Z)]:
    print(name, np.allclose(U.conj().T @ U, np.eye(2)))

## 15. Operador Hadamard

Hadamard se define por

$$
H=\frac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}.
$$

Aplicado a $\left|0\right\rangle$, produce un vector normalizado con dos amplitudes iguales.

In [ ]:
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
print("H|0> =", H @ ket0)
print("H|1> =", H @ ket1)
print("norma de H|0>:", np.vdot(H @ ket0, H @ ket0))

## 16. Operador Pauli $Y$

El operador

$$
Y=\begin{pmatrix}0&-i\\ i&0\end{pmatrix}
$$

actúa sobre $(\alpha,\beta)^T$ como

$$
Y(\alpha,\beta)^T=(-i\beta,i\alpha)^T.
$$

In [ ]:
state = np.array([1j/np.sqrt(6), (2+1j)/np.sqrt(6)], dtype=complex)
result = Y @ state
print(result)
expected = np.array([(1-2j)/np.sqrt(6), -1/np.sqrt(6)], dtype=complex)
print("coincide con el cálculo manual:", np.allclose(result, expected))

## 17. Ejercicio guiado: normalización

Sea

$$
v=\begin{pmatrix}1-i\\ \sqrt2\end{pmatrix}.
$$

Entonces

$$
\|v\|^2=|1-i|^2+|\sqrt2|^2=2+2=4,
$$

por lo que el vector normalizado es $v/2$.

In [ ]:
v = np.array([1-1j, np.sqrt(2)], dtype=complex)
v_norm = v / np.sqrt(np.vdot(v, v).real)
print(v_norm)
print(np.vdot(v_norm, v_norm))

## 18. Ejercicio guiado: posible valor de una amplitud

Si

$$
\left|\psi\right\rangle=\begin{pmatrix}a\\(1-i)/2\end{pmatrix}
$$

es válido, entonces

$$
|a|^2+\left|\frac{1-i}{2}\right|^2=1.
$$

Como el segundo término vale $1/2$, se requiere $|a|^2=1/2$.

In [ ]:
candidates = {
    "(i-1)/2": (1j - 1)/2,
    "i/2": 1j/2,
    "(i+1)/4": (1j + 1)/4,
    "-i/sqrt(2)": -1j/np.sqrt(2),
}
for label, a in candidates.items():
    print(label, "|a|^2 =", abs(a)**2)

## 19. Visualización de amplitudes válidas

La condición $|a|^2=1/2$ significa que $a$ debe estar sobre un círculo de radio $1/\sqrt2$ en el plano complejo. Esta representación deja claro que existen infinitos valores matemáticamente posibles.

In [ ]:
radius = 1/np.sqrt(2)
angles = np.linspace(0, 2*np.pi, 300)
points = radius * np.exp(1j * angles)
fig, ax = plt.subplots()
ax.plot(points.real, points.imag)
for label, a in candidates.items():
    ax.scatter(a.real, a.imag)
    ax.text(a.real + 0.03, a.imag + 0.03, label)
ax.axhline(0, linewidth=1); ax.axvline(0, linewidth=1)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Valores de a con |a|² = 1/2")
ax.set_xlabel("Parte real")
ax.set_ylabel("Parte imaginaria")
plt.show()

## 20. Ejercicios propuestos

1. Convertir $-2-2i$ a forma polar principal y dibujarlo.
2. Calcular el conjugado de $3-4i$ y verificar que $z\overline z=|z|^2$.
3. Normalizar el vector $(2+i,1-i)^T$.
4. Verificar con NumPy si $\frac{1}{\sqrt2}\begin{pmatrix}1&-i\\1&i\end{pmatrix}$ es unitaria.
5. Construir un estado de qubit normalizado y graficar sus amplitudes y probabilidades.

## 21. Resumen final

Los números complejos permiten describir simultáneamente magnitud y dirección. La forma rectangular facilita componentes; la forma polar y exponencial facilitan geometría y multiplicación. En álgebra lineal compleja, la conjugación es indispensable para definir producto interno, norma y adjunta. En computación cuántica elemental, los estados son vectores complejos normalizados, las probabilidades se obtienen con la regla de Born y las evoluciones ideales se modelan con matrices unitarias.